# BAH Dataset — Text Training (BERT)

Trains a BERT-based sequence classifier on interview transcripts.

**Modality:** Text transcripts stored in the video-level split files.

**Pipeline:**
1. Load transcripts via `get_text_splits()` with a HuggingFace tokenizer
2. Fine-tune `BertForSequenceClassification` (or any AutoModel)
3. Evaluate on val split after every epoch; save best weights by macro-F1
4. Final inference on test split

In [ ]:
import os, sys, random, warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_cosine_schedule_with_warmup,
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
)
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

# ── Paths — run from repo root or adjust sys.path ────────────────────────────
REPO_ROOT = Path('..').resolve()          # text/src/ → repo root
sys.path.insert(0, str(REPO_ROOT / 'utils'))
from load_dataset import get_text_splits, CLASS_NAMES, NUM_CLASSES

# ── Config ────────────────────────────────────────────────────────────────────
MODEL_NAME   = 'bert-base-uncased'
MAX_LENGTH   = 512
EPOCHS       = 10
BATCH_SIZE   = 16
LR           = 2e-5
WD           = 1e-4
NUM_WORKERS  = 0
SEED         = 42
PATIENCE     = 3          # early-stopping patience (0 = disabled)
WARMUP_RATIO = 0.1

WEIGHTS_DIR  = Path('../weights')
WEIGHTS_NAME = 'bert_bah_best.pth'
LAST_NAME    = 'bert_bah_last.pth'

# ── Reproducibility ───────────────────────────────────────────────────────────
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ── Device ────────────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')

print(f'Device  : {DEVICE}')
print(f'Model   : {MODEL_NAME}')
print(f'Epochs  : {EPOCHS}  |  Batch : {BATCH_SIZE}  |  LR : {LR}')
print(f'Max len : {MAX_LENGTH} tokens')

In [ ]:
# ── Tokenizer & datasets ──────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

splits = get_text_splits(max_length=MAX_LENGTH, tokenizer=tokenizer)
train_ds, val_ds, test_ds = splits['train'], splits['val'], splits['test']

# ── DataLoaders ───────────────────────────────────────────────────────────────
_pw = NUM_WORKERS > 0
_pm = DEVICE.type == 'cuda'

def collate_fn(batch):
    """Collate tokenised-dict + label pairs into batched tensors."""
    features, labels = zip(*batch)
    keys = features[0].keys()
    batched = {k: torch.stack([f[k] for f in features]) for k in keys}
    return batched, torch.tensor(labels, dtype=torch.long)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=_pm,
    persistent_workers=_pw, collate_fn=collate_fn, drop_last=True,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=_pm,
    persistent_workers=_pw, collate_fn=collate_fn,
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=_pm,
    persistent_workers=_pw, collate_fn=collate_fn,
)

print(f'Train batches : {len(train_loader):,}')
print(f'Val   batches : {len(val_loader):,}')
print(f'Test  batches : {len(test_loader):,}')

# ── Sanity check: show a few samples ─────────────────────────────────────────
print('\nSample transcripts:')
for i in range(min(3, len(train_ds))):
    # Access raw text before tokenisation by using the underlying data
    row = train_ds.data.iloc[i]
    print(f'  [{CLASS_NAMES[row["label"]]}] {row["transcript"][:120]}...')

In [ ]:
# ── Model ─────────────────────────────────────────────────────────────────────
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True,
).to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params     : {total_params:,}')
print(f'Trainable params : {trainable_params:,}')

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)

total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps,
)
print(f'Total steps: {total_steps:,}  |  Warmup: {warmup_steps:,}')

In [ ]:
# ── Training loop ─────────────────────────────────────────────────────────────
def compute_metrics(y_true, y_pred):
    return {
        'acc'      : accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'recall'   : recall_score(y_true, y_pred, average='macro', zero_division=0),
        'f1'       : f1_score(y_true, y_pred, average='macro', zero_division=0),
    }


def run_epoch(loader, model, criterion, optimizer=None, scheduler=None, desc=''):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []

    pbar = tqdm(loader, desc=desc, leave=False, unit='batch')
    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for features, labels in pbar:
            features = {k: v.to(DEVICE) for k, v in features.items()}
            labels   = labels.to(DEVICE)

            outputs = model(**features)
            logits  = outputs.logits
            loss    = criterion(logits, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                if scheduler is not None:
                    scheduler.step()

            total_loss += loss.item() * len(labels)
            all_preds.extend(logits.argmax(1).cpu().tolist())
            all_labels.extend(labels.cpu().tolist())
            pbar.set_postfix(loss=f'{loss.item():.4f}')

    avg_loss = total_loss / len(loader.dataset)
    return avg_loss, compute_metrics(all_labels, all_preds)


WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

history     = defaultdict(list)
best_val_f1 = -1.0
no_improve  = 0

header = (
    f"{'Epoch':>6} {'TrLoss':>8} {'TrAcc':>7} {'TrF1':>7} "
    f"{'VaLoss':>8} {'VaAcc':>7} {'VaPrec':>8} {'VaRec':>7} {'VaF1':>7} {'LR':>10}"
)
print(f'Starting training for {EPOCHS} epochs on {DEVICE}...')
print('─' * len(header))
print(header)
print('─' * len(header))

epoch_bar = tqdm(range(1, EPOCHS + 1), desc='Epochs', unit='epoch')
for epoch in epoch_bar:
    tr_loss, tr_m = run_epoch(
        train_loader, model, criterion, optimizer, scheduler,
        desc=f'Ep {epoch}/{EPOCHS} train',
    )
    va_loss, va_m = run_epoch(
        val_loader, model, criterion,
        desc=f'Ep {epoch}/{EPOCHS} val  ',
    )
    epoch_bar.set_postfix(val_f1=f'{va_m["f1"]:.4f}', val_acc=f'{va_m["acc"]:.4f}')

    cur_lr = optimizer.param_groups[0]['lr']

    history['epoch'].append(epoch)
    history['tr_loss'].append(tr_loss)
    history['va_loss'].append(va_loss)
    history['lr'].append(cur_lr)
    for k, v in tr_m.items():
        history[f'tr_{k}'].append(v)
    for k, v in va_m.items():
        history[f'va_{k}'].append(v)

    print(
        f"{epoch:>6} {tr_loss:>8.4f} {tr_m['acc']:>7.4f} {tr_m['f1']:>7.4f} "
        f"{va_loss:>8.4f} {va_m['acc']:>7.4f} {va_m['precision']:>8.4f} "
        f"{va_m['recall']:>7.4f} {va_m['f1']:>7.4f} {cur_lr:>10.2e}"
    )

    if va_m['f1'] > best_val_f1:
        best_val_f1 = va_m['f1']
        torch.save(model.state_dict(), WEIGHTS_DIR / WEIGHTS_NAME)
        print(f'  New best val F1={best_val_f1:.4f} — saved {WEIGHTS_NAME}')
        no_improve = 0
    else:
        no_improve += 1

    if PATIENCE > 0 and no_improve >= PATIENCE:
        print(f'Early stopping after {epoch} epochs (patience={PATIENCE})')
        break

print('─' * len(header))
print(f'Training complete.  Best val F1 = {best_val_f1:.4f}')
torch.save(model.state_dict(), WEIGHTS_DIR / LAST_NAME)
print(f'Last-epoch weights saved -> {WEIGHTS_DIR / LAST_NAME}')

In [ ]:
# ── Training curves ───────────────────────────────────────────────────────────
hist = pd.DataFrame(history)

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

axes[0, 0].plot(hist['epoch'], hist['tr_loss'], label='Train')
axes[0, 0].plot(hist['epoch'], hist['va_loss'], label='Val')
axes[0, 0].set_title('Loss')
axes[0, 0].legend()
axes[0, 0].set_xlabel('Epoch')

axes[0, 1].plot(hist['epoch'], hist['tr_acc'], label='Train')
axes[0, 1].plot(hist['epoch'], hist['va_acc'], label='Val')
axes[0, 1].set_title('Accuracy')
axes[0, 1].legend()
axes[0, 1].set_xlabel('Epoch')

axes[1, 0].plot(hist['epoch'], hist['tr_f1'], label='Train')
axes[1, 0].plot(hist['epoch'], hist['va_f1'], label='Val')
axes[1, 0].set_title('F1 (macro)')
axes[1, 0].legend()
axes[1, 0].set_xlabel('Epoch')

axes[1, 1].plot(hist['epoch'], hist['va_precision'], label='Precision')
axes[1, 1].plot(hist['epoch'], hist['va_recall'],    label='Recall')
axes[1, 1].plot(hist['epoch'], hist['va_f1'],        label='F1')
axes[1, 1].set_title('Val Precision / Recall / F1')
axes[1, 1].legend()
axes[1, 1].set_xlabel('Epoch')

for ax in axes.flat:
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(alpha=0.3)

plt.suptitle('Text (BERT) — Training history', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Test inference ────────────────────────────────────────────────────────────
best_ckpt = WEIGHTS_DIR / WEIGHTS_NAME
model.load_state_dict(torch.load(best_ckpt, map_location=DEVICE))
print(f'Loaded best weights from {best_ckpt}')

model.eval()
all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for features, labels in tqdm(test_loader, desc='Test inference'):
        features = {k: v.to(DEVICE) for k, v in features.items()}
        logits   = model(**features).logits
        probs    = torch.softmax(logits, dim=1).cpu().numpy()
        preds    = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.numpy().tolist())
        all_probs.extend(probs.tolist())

test_m = compute_metrics(all_labels, all_preds)

print()
print('=' * 50)
print('TEST SET METRICS')
print('=' * 50)
print(f"  Accuracy  : {test_m['acc']:.4f}")
print(f"  Precision : {test_m['precision']:.4f}  (macro)")
print(f"  Recall    : {test_m['recall']:.4f}  (macro)")
print(f"  F1        : {test_m['f1']:.4f}  (macro)")
print()
print('Per-class report:')
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES, digits=4))

# ── Confusion matrix ──────────────────────────────────────────────────────────
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(NUM_CLASSES))
ax.set_yticks(range(NUM_CLASSES))
ax.set_xticklabels(CLASS_NAMES)
ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Confusion Matrix — Test Set', fontweight='bold')
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(j, i, f'{cm[i, j]:,}', ha='center', va='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black', fontsize=12)
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()